<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/git_man.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone --depth 1 https://github.com/git/git.git


Cloning into 'git'...
remote: Enumerating objects: 4933, done.
remote: Counting objects: 100% (4933/4933), done.
remote: Compressing objects: 100% (4323/4323), done.
remote: Total 4933 (delta 498), reused 2167 (delta 439), pack-reused 0 (from 0)
Receiving objects: 100% (4933/4933), 12.35 MiB | 4.07 MiB/s, done.
Resolving deltas: 100% (498/498), done.
0 command doc files found
[]


In [2]:
import os
doc_files = [f for f in os.listdir('git/Documentation') if f.startswith('git-') and f.endswith('.adoc')]
print(len(doc_files), "command doc files found")
print(doc_files[:10])

169 command doc files found
['git-rm.adoc', 'git-rebase.adoc', 'git-request-pull.adoc', 'git-unpack-file.adoc', 'git-difftool.adoc', 'git-check-attr.adoc', 'git-prune-packed.adoc', 'git-verify-commit.adoc', 'git-show.adoc', 'git-branch.adoc']


In [3]:
with open('git/Documentation/git-rebase.adoc', 'r') as f:
    content = f.read()

print(content[:2000])

git-rebase(1)

NAME
----
git-rebase - Reapply commits on top of another base tip

SYNOPSIS
--------
[verse]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>]
	[--onto <newbase> | --keep-base] [<upstream> [<branch>]]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>] [--onto <newbase>]
	--root [<branch>]
'git rebase' (--continue|--skip|--abort|--quit|--edit-todo|--show-current-patch)

DESCRIPTION
-----------
Transplant a series of commits onto a different starting point.
You can also use `git rebase` to reorder or combine commits: see INTERACTIVE
MODE below for how to do that.

For example, imagine that you have been working on the `topic` branch in this
history, and you want to "catch up" to the work done on the `master` branch.

------------
          A---B---C topic
         /
    D---E---F---G master
------------

You want to transplant the commits you made on `topic` since it diverged from
`master` (i.e. A, B, and C), on top of the current `master`.  You can do

In [4]:
import re

def parse_adoc_file(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    cmd_name = os.path.basename(filepath).replace('.adoc', '')

    # Match: an all-caps header line, followed by a line of dashes
    pattern = re.compile(r'^([A-Z][A-Z \-]{2,})\n-{3,}\n', re.MULTILINE)

    matches = list(pattern.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        header = m.group(1).strip()
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{header}\n{body}",
                "metadata": {"command": cmd_name, "section": header}
            })

    return chunks

In [5]:
chunks = parse_adoc_file('git/Documentation/git-rebase.adoc')
print(len(chunks), "chunks found")
for c in chunks:
    print("---", c['metadata']['section'], "---")
    print(c['text'][:150])
    print()

15 chunks found
--- NAME ---
NAME
git-rebase - Reapply commits on top of another base tip

--- SYNOPSIS ---
SYNOPSIS
[verse]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>]
	[--onto <newbase> | --keep-base] [<upstream> [<branch>]]
'git rebase' [

--- DESCRIPTION ---
DESCRIPTION
Transplant a series of commits onto a different starting point.
You can also use `git rebase` to reorder or combine commits: see INTERACTI

--- TRANSPLANTING A TOPIC BRANCH WITH --ONTO ---
TRANSPLANTING A TOPIC BRANCH WITH --ONTO
Here is how you would transplant a topic branch based on one
branch to another, to pretend that you forked th

--- MODE OPTIONS ---
MODE OPTIONS
The options in this section cannot be used with any other option,
including not with each other:

--continue::
	Restart the rebasing proc

--- OPTIONS ---
OPTIONS
--onto <newbase>::
	Starting point at which to create the new commits. If the
	`--onto` option is not specified, the starting point is
	`<upst

--- INCOMPATIBLE OPTIONS -

In [8]:
SKIP_SECTIONS = {"GIT"}  # boilerplate footer, no useful content

def parse_adoc_file(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    cmd_name = os.path.basename(filepath).replace('.adoc', '')

    # Match: an all-caps header line, followed by a line of dashes
    pattern = re.compile(r'^([A-Z][A-Z \-]{2,})\n-{3,}\n', re.MULTILINE)

    matches = list(pattern.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        header = m.group(1).strip()
        if header in SKIP_SECTIONS:
            continue
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{header}\n{body}",
                "metadata": {"command": cmd_name, "section": header}
            })

    return chunks

In [9]:
all_chunks = []
for f in doc_files:
    all_chunks.extend(parse_adoc_file(os.path.join('git/Documentation', f)))

print(len(all_chunks), "total chunks across all commands")

# sanity check: distribution of section types
from collections import Counter
section_counts = Counter(c['metadata']['section'] for c in all_chunks)
print(section_counts.most_common(15))

1135 total chunks across all commands
[('NAME', 167), ('SYNOPSIS', 167), ('DESCRIPTION', 167), ('OPTIONS', 145), ('EXAMPLES', 72), ('SEE ALSO', 72), ('CONFIGURATION', 50), ('OUTPUT', 17), ('DISCUSSION', 16), ('COMMANDS', 15), ('BUGS', 10), ('NOTES', 10), ('CAVEATS', 9), ('FILES', 8), ('ENVIRONMENT', 6)]


In [10]:
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="git_docs")

texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"{c['metadata']['command']}_{c['metadata']['section']}_{i}" for i, c in enumerate(all_chunks)]

embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(collection.count(), "chunks added to Chroma")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

1135 chunks added to Chroma


In [13]:
query = "how do I undo my last commit"
query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    print(f"[{meta['command']} / {meta['section']}] (dist={dist:.3f})")
    print(doc[:200])
    print()

[git-revert / SYNOPSIS] (dist=0.769)
SYNOPSIS
[verse]
'git revert' [--[no-]edit] [-n] [-m <parent-number>] [-s] [-S[<keyid>]] <commit>...
'git revert' (--continue | --skip | --abort | --quit)

[git-revert / NAME] (dist=0.818)
NAME
git-revert - Revert some existing commits

[git-revert / DESCRIPTION] (dist=0.879)
DESCRIPTION
Given one or more existing commits, revert the changes that the
related patches introduce, and record some new commits that record
them.  This requires your working tree to be clean (no mo

[git-reset / DESCRIPTION] (dist=0.906)
DESCRIPTION
`git reset` does either of the following:

1. `git reset [<mode>] <commit>` changes which commit `HEAD` points to. This
   makes it possible to undo various Git operations, for example com

[git-revert / OPTIONS] (dist=0.944)
OPTIONS
<commit>...::
	Commits to revert.
	For a more complete list of ways to spell commit names, see
	linkgit:gitrevisions[7].
	Sets of commits can also be given but no traversal is done by
	default

